In [0]:
# --- Core data libraries ---
import pandas as pd
import numpy as np
import json
import os
import shutil

# --- HTTP / external API calls ---
import requests

# --- PySpark (work with imported data in the lakehouse) ---
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

print("All imports loaded successfully.")

In [0]:
folder_path = "/Volumes/workspace/nj_transit_bronze/kaggle_data/"

df_raw_trips = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(folder_path)
)

df_raw_trips.show()

In [0]:
df_raw_trips.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.nj_transit_bronze.transit_trips")

print("Delta table saved: workspace.nj_transit_bronze.transit_trips")

# Verify the table was written
print(f"Row count: {spark.table('workspace.nj_transit_bronze.transit_trips').count()}")

In [0]:
api_url_stations = 'https://services6.arcgis.com/M0t0HPE53pFK525U/arcgis/rest/services/NJTransit_Rail_Stations/FeatureServer/0/query'

params = {
    "where": "1=1",
    "outFields": "*",
    "f": "geojson",
    "returnGeometry": "true"
}

response = requests.get(api_url_stations, params=params)

print(f"Status Code: {response.status_code}")

data = response.json()

# Extract features from GeoJSON response and convert to DataFrame
features = data.get('features', [])

# Each feature has 'properties' (attributes) and 'geometry' (coordinates)
records = [f['properties'] for f in features if 'properties' in f]
df_stations = spark.createDataFrame(pd.DataFrame(records))



In [0]:
df_stations.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.nj_transit_bronze.stations")

print("Delta table saved: workspace.nj_transit_bronze.transit_stations")

In [0]:
base_line_endpoint = 'https://services6.arcgis.com/M0t0HPE53pFK525U/arcgis/rest/services/NJTransit_Rail_Linea/FeatureServer'

records = []
for i in range(0, 14):

    current_endpoint = f"{base_line_endpoint}/{i}/query"

    params = {
        "where": "1=1",
        "outFields": "*",
        "f": "geojson",
        "returnGeometry": "true"
    }

    response = requests.get(current_endpoint, params=params)

    print(f"Status Code: {response.status_code}")

    data = response.json()

    features = data.get("features", [])

    for f in features:
      props = f.get("properties", {}) or {}
      geom = f.get("geometry", {}) or {}

      # Combine properties and geometry into a single flat dictionary record
      record = {
          **props,
          "geometry_type": geom.get("type"),
          "coordinates": str(
              geom.get("coordinates")
          ),  # Convert coordinate list/array to string for safe Spark handling
      }
      records.append(record)

# Load into Spark
df_lines = spark.createDataFrame(pd.DataFrame(records))
df_lines.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.nj_transit_bronze.lines")

In [0]:
weather_url = "https://archive-api.open-meteo.com/v1/archive"

nec_locations = [
    {"region": "New York City (NY)", "latitude": 40.7128, "longitude": -74.0060},
    {"region": "Newark (NJ)", "latitude": 40.7357, "longitude": -74.1724},
    {"region": "Trenton (NJ)", "latitude": 40.2206, "longitude": -74.7597},
]

start_date = '2018-01-01'
end_date = '2020-12-31'

all_weather_list = []

for loc in nec_locations:

    params = {
        "latitude": loc['latitude'],
        "longitude": loc['longitude'],
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,precipitation,rain,wind_speed_10m",
        "timezone": "America/New_York"
    }

    response = requests.get(weather_url, params=params)
    data = response.json()

    hourly = data.get("hourly", {})

    times = hourly.get("time", [])
    temps = hourly.get("temperature_2m", [])
    precips = hourly.get("precipitation", [])
    winds = hourly.get("wind_speed_10m", [])

    for ti, te, pr, wi in zip(times, temps, precips, winds):

        all_weather_list.append({
          "region_name": loc["region"],
          "latitude": loc["latitude"],
          "longitude": loc["longitude"],
          "timestamp": ti,
          "temperature_2m_c": te,
          "precipitation_mm": pr,
          "wind_speed_kmh": wi,
      })
        
df_weather = spark.createDataFrame(pd.DataFrame(all_weather_list))
df_weather.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.nj_transit_bronze.weather")